# Проверка API-ручек Map App

Запусти бэкенд перед выполнением ячеек:
```bash
cd app/backend
uvicorn app.main:app --reload
```

In [5]:
import requests
import json

BASE = "http://localhost:8000"

def pp(data):
    """Pretty-print JSON."""
    print(json.dumps(data, indent=2, ensure_ascii=False))

def safe_json(r):
    """Попытка распарсить JSON с диагностикой."""
    if r.status_code >= 400:
        print(f"❌ HTTP {r.status_code}")
        print(f"Headers: {dict(r.headers)}")
        print(f"Body: {r.text[:500]}")
        return None
    try:
        return r.json()
    except Exception as e:
        print(f"❌ JSON error: {e}")
        print(f"Status: {r.status_code}")
        print(f"Content-Type: {r.headers.get('content-type', 'unknown')}")
        print(f"Body: {r.text[:500]}")
        return None

## Health Check

In [9]:
r = requests.get(f"{BASE}/health")
print(f"Status: {r.status_code}")
pp(r.json())

Status: 200
{
  "status": "ok"
}


## 1. Корпуса

In [10]:
r = requests.get(f"{BASE}/buildings")
print(f"Status: {r.status_code}")
buildings = safe_json(r)
if buildings is None:
    print("\n⚠️ Сервер не запустился или недоступен. Запусти:")
    print("   cd app/backend")
    print("   uvicorn app.main:app --reload")
else:
    print(f"Всего корпусов: {len(buildings)}")
    for b in buildings[:5]:
        print(f"  {b['short_name']:6s} — {b['name']}")

    # Сохраним ID корпуса для следующих запросов
    BUILDING_ID = buildings[0]["id"]
    print(f"\nВыбранный корпус: {buildings[0]['name']} ({BUILDING_ID})")

Status: 200
Всего корпусов: 32
  1      — Восточное крыло главного корпуса
  ДС     — Дворец спорта
  1      — Западное крыло главного корпуса
  10М    — Корпус 10М, ул. Марата, 8 (ФВО)
  11Ч    — Корпус 11Ч, ул. Чайковского, 20а

Выбранный корпус: Восточное крыло главного корпуса (bce65150-c95a-40cb-a59e-9774f5e1b249)


In [11]:
# Один корпус по ID
r = requests.get(f"{BASE}/buildings/{BUILDING_ID}")
print(f"Status: {r.status_code}")
data = safe_json(r)
if data: pp(data)

Status: 200
{
  "id": "bce65150-c95a-40cb-a59e-9774f5e1b249",
  "name": "Восточное крыло главного корпуса",
  "short_name": "1"
}


## 2. Этажи

In [12]:
r = requests.get(f"{BASE}/buildings/{BUILDING_ID}/floors")
print(f"Status: {r.status_code}")
floors = safe_json(r) or []
print(f"Этажей: {len(floors)}")
for f in floors:
    print(f"  Этаж {f['floor_number']}")

Status: 200
Этажей: 5
  Этаж 1
  Этаж 2
  Этаж 3
  Этаж 4
  Этаж 5


## 3. Комнаты

In [13]:
FLOOR_NUM = floors[0]["floor_number"]
print(f"Комнаты на этаже {FLOOR_NUM} корпуса {BUILDING_ID[:8]}...")

r = requests.get(f"{BASE}/buildings/{BUILDING_ID}/floors/{FLOOR_NUM}/rooms")
print(f"Status: {r.status_code}")
rooms = safe_json(r) or []
print(f"Всего комнат: {len(rooms)}")
for room in rooms[:5]:
    print(f"  №{room['number']:6s} — {room['name'] or room['room_type'] or 'без имени'}")

# Сохраняем ID комнат для построения пути
if len(rooms) >= 2:
    ROOM_START = rooms[0]["id"]
    ROOM_END = rooms[1]["id"]
    print(f"\nКомната A: №{rooms[0]['number']} ({ROOM_START})")
    print(f"Комната B: №{rooms[1]['number']} ({ROOM_END})")

Комнаты на этаже 1 корпуса bce65150...
Status: 200
Всего комнат: 34
  №163    — Учебная аудитория
  №145    — Лаборатория 
  №146    — Учебная лаборатория "Автоматизированный электропривод"
  №165а   — Учебная лаборатория направления "Политология"
  №162    — Учебная аудитория

Комната A: №163 (b8de0b64-9ca3-7b6c-2270-a7db5e355439)
Комната B: №145 (a646c72f-b0d6-4442-addd-9f9997af9642)


## 4. Технические помещения (лестницы, лифты, туалеты...)

In [14]:
r = requests.get(f"{BASE}/buildings/{BUILDING_ID}/floors/{FLOOR_NUM}/technical")
print(f"Status: {r.status_code}")
technical = safe_json(r) or []
print(f"Всего тех. помещений: {len(technical)}")
for t in technical:
    linked_info = f", linked: {len(t['linked'])}" if t.get("linked") else ""
    print(f"  {t['type']:12s} — {t['name'] or 'без имени'} | has_entrance: {t['has_entrance']}{linked_info}")

Status: 200
Всего тех. помещений: 8
  Гардероб     — Гардероб | has_entrance: False
  Охрана       — Охрана | has_entrance: False
  Лестница     — Лестница | has_entrance: True
  Лестница     — Лестница | has_entrance: True
  Лестница     — Лестница | has_entrance: True
  Лестница     — Лестница | has_entrance: True
  Туалет       — Туалет женский | has_entrance: False
  Туалет       — Туалет мужской | has_entrance: False


## 5. Входы/выходы

In [15]:
r = requests.get(f"{BASE}/buildings/{BUILDING_ID}/floors/{FLOOR_NUM}/entrances")
print(f"Status: {r.status_code}")
entrances = safe_json(r) or []
print(f"Всего входов: {len(entrances)}")
for e in entrances:
    print(f"  {e['object_type']:10s} — {e['room_number']} | ({e['x']}, {e['y']})")

Status: 200
Всего входов: 36
  room       — 141 | (90, 171)
  room       — 143 | (106, 365)
  room       — 145 | (106, 414)
  stairs     — Лестница 1 | (120, 194)
  room       — 147 | (106, 518)
  room       — 149 | (106, 578)
  stairs     — Лестница 1 | (168, 727)
  room       — 151 | (350, 748)
  room       — 146 | (309, 788)
  room       — 153 | (521, 748)
  room       — 153а | (624, 748)
  room       — 146а | (619, 788)
  room       — 155 | (825, 748)
  room       — 148 | (825, 788)
  room       — 150 | (887, 788)
  room       — 155а | (865, 748)
  room       — 152 | (932, 788)
  room       — 154 | (1033, 768)
  room       — 156 | (1033, 684)
  stairs     — Лестница 1 | (950, 709)
  room       — 158 | (1030, 626)
  room       — 157 | (989, 562)
  room       — 160 | (1030, 465)
  room       — 159 | (989, 526)
  room       — 161 | (989, 463)
  room       — 162 | (1030, 412)
  room       — 163 | (989, 413)
  room       — 165 | (989, 360)
  room       — 162а | (1030, 363)
  room       

## 6. Сетка навигации (grid)

In [16]:
r = requests.get(f"{BASE}/buildings/{BUILDING_ID}/floors/{FLOOR_NUM}/grid")
print(f"Status: {r.status_code}")
grid = safe_json(r)
if grid:
    print(f"cell_size: {grid['cell_size']}")
    print(f"nodes: {len(grid['nodes'])}")
    print(f"edges: {len(grid['edges'])}")
    print(f"entrance_connections: {len(grid['entrance_connections'])}")

Status: 200
cell_size: 21
nodes: 289
edges: 384
entrance_connections: 36


## 7. Построение пути

In [20]:
# Путь между двумя комнатами на одном этаже
if len(rooms) >= 2:
    payload = {
        "building_id": BUILDING_ID,
        "start_object_id": ROOM_START,
        "end_object_id": ROOM_END,
    }
    print(f"Путь: комната {rooms[0]['number']} → {rooms[1]['number']}")
    r = requests.post(f"{BASE}/path", json=payload)
    print(f"Status: {r.status_code}")
    data = safe_json(r)
    if data: pp(data)

Путь: комната 163 → 145
Status: 200
{
  "found": true,
  "path": [
    {
      "floor_number": "1",
      "nodes": [
        {
          "x": 989,
          "y": 413
        },
        {
          "x": 1015,
          "y": 413
        },
        {
          "x": 1015,
          "y": 430
        },
        {
          "x": 1015,
          "y": 451
        },
        {
          "x": 1015,
          "y": 463
        },
        {
          "x": 1015,
          "y": 465
        },
        {
          "x": 1015,
          "y": 472
        },
        {
          "x": 1015,
          "y": 493
        },
        {
          "x": 1015,
          "y": 514
        },
        {
          "x": 1015,
          "y": 526
        },
        {
          "x": 1015,
          "y": 535
        },
        {
          "x": 1015,
          "y": 556
        },
        {
          "x": 1015,
          "y": 562
        },
        {
          "x": 1015,
          "y": 577
        },
        {
          "x": 1015,

In [18]:
# Ошибка: разные корпуса
if len(buildings) >= 2:
    other_building_id = buildings[1]["id"]
    # Берём первую комнату из другого корпуса
    r2 = requests.get(f"{BASE}/buildings/{other_building_id}/floors/1/rooms")
    if r2.status_code == 200 and r2.json():
        other_room_id = r2.json()[0]["id"]
        payload = {
            "building_id": BUILDING_ID,
            "start_object_id": ROOM_START,
            "end_object_id": other_room_id,
        }
        print(f"Путь между РАЗНЫМИ корпусами")
        r = requests.post(f"{BASE}/path", json=payload)
        print(f"Status: {r.status_code}")
        data = safe_json(r)
        if data: pp(data)

Путь между РАЗНЫМИ корпусами
Status: 200
{
  "found": false,
  "path": [],
  "total_length": 0.0,
  "floor_transitions": [],
  "error": "Objects are in different buildings"
}


In [19]:
# Ошибка: несуществующая комната
payload = {
    "building_id": BUILDING_ID,
    "start_object_id": "00000000-0000-0000-0000-000000000000",
    "end_object_id": ROOM_END,
}
print(f"Путь с несуществующей комнатой")
r = requests.post(f"{BASE}/path", json=payload)
print(f"Status: {r.status_code}")
data = safe_json(r)
if data: pp(data)

Путь с несуществующей комнатой
Status: 404
❌ HTTP 404
Headers: {'date': 'Tue, 07 Apr 2026 17:14:00 GMT', 'server': 'uvicorn', 'content-length': '33', 'content-type': 'application/json'}
Body: {"detail":"Start room not found"}
